<a href="https://colab.research.google.com/github/ridamumtazz/Flyrank-ML-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 1. My rule and its reason codes

### My Rule

I will prioritize content items that have strong search visibility but appear to have an opportunity for improvement. The baseline action score will give higher priority to content with high Google Search impressions, lower click-through rate (CTR), and an average search position where improvement may be possible.

This score is a directional decision-support tool based on the available June 2026 data. It does not guarantee that a content item needs improvement. The highest-ranked items should be reviewed manually before taking action.

### Reason Codes

* **HIGH_IMPRESSIONS_LOW_CTR** — The content receives many search impressions but gets relatively few clicks.
* **RANKING_OPPORTUNITY** — The content has an average search position where additional optimization may help improve visibility.
* **HIGH_SEARCH_VISIBILITY** — The content has strong search visibility based on impressions.
* **LOW_ENGAGEMENT** — The content has relatively low engagement compared with its sessions.

### Action Labels

* **Improve** — Strong opportunity signals suggest the content should be reviewed for optimization.
* **Review** — Some opportunity signals are present, but more manual checking is needed.
* **Monitor** — The content does not show a strong enough signal for immediate action.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [37]:
import os

print(os.path.exists('work/outputs/baseline_action_score.csv'))

True


In [29]:
import os

print(os.listdir('.'))

['.config', 'flyrank-ml-internship-starter', 'work', 'sample_data']


In [30]:
import pandas as pd

df = pd.read_csv(
    'flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv'
)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print(df.columns.tolist())


Dataset loaded successfully!
Shape: (30000, 44)
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [31]:
!git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git

import os

for root, dirs, files in os.walk('flyrank-ml-internship-starter'):
    for file in files:
        if 'content_refresh' in file.lower() and file.endswith('.csv'):
            print(os.path.join(root, file))

fatal: destination path 'flyrank-ml-internship-starter' already exists and is not an empty directory.
flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv


In [32]:
# ============================================================
# 2. BUILD THE RANKED QUEUE
# ML-07 — Baseline Action Score and Top-20 Review
# ============================================================

import pandas as pd
import numpy as np
import os

# ------------------------------------------------------------
# Step 1: Create a working copy
# ------------------------------------------------------------

content_data = df.copy()

print("Starting dataset shape:", content_data.shape)


# ------------------------------------------------------------
# Step 2: Create safe scoring components
# ------------------------------------------------------------

# High impressions = more search visibility and larger potential opportunity
# log1p reduces the effect of extremely large values

log_impressions = np.log1p(
    content_data['impressions_90d'].fillna(0)
)

max_log_impressions = log_impressions.max()

if max_log_impressions > 0:
    content_data['impression_score'] = (
        log_impressions / max_log_impressions
    )
else:
    content_data['impression_score'] = 0


# ------------------------------------------------------------
# Step 3: Create low CTR score
# ------------------------------------------------------------

# Lower CTR means more potential opportunity for improvement

ctr_values = content_data['ctr'].fillna(0)

max_ctr = ctr_values.max()

if max_ctr > 0:
    content_data['low_ctr_score'] = (
        1 - (ctr_values / max_ctr)
    )
else:
    content_data['low_ctr_score'] = 0

content_data['low_ctr_score'] = (
    content_data['low_ctr_score'].clip(0, 1)
)


# ------------------------------------------------------------
# Step 4: Create ranking opportunity score
# ------------------------------------------------------------

# Positions 4–20 are treated as a possible optimization opportunity

content_data['ranking_opportunity_score'] = np.where(
    (content_data['avg_position'] >= 4) &
    (content_data['avg_position'] <= 20),
    1,
    0
)


# ------------------------------------------------------------
# Step 5: Create trend score
# ------------------------------------------------------------

# Positive or stable trend is not used as a reason to improve.
# Negative trend is treated as an additional directional signal.

content_data['trend_score'] = np.where(
    content_data['trend_direction'].astype(str).str.lower().isin(
        ['down', 'declining', 'decrease']
    ),
    1,
    0
)


# ------------------------------------------------------------
# Step 6: Calculate the Baseline Action Score
# ------------------------------------------------------------

content_data['action_score'] = (
    0.35 * content_data['impression_score'] +
    0.30 * content_data['low_ctr_score'] +
    0.25 * content_data['ranking_opportunity_score'] +
    0.10 * content_data['trend_score']
) * 100


# ------------------------------------------------------------
# Step 7: Assign Action Labels
# ------------------------------------------------------------

content_data['action'] = np.select(
    [
        content_data['action_score'] >= 70,
        content_data['action_score'] >= 40
    ],
    [
        'Improve',
        'Review'
    ],
    default='Monitor'
)


# ------------------------------------------------------------
# Step 8: Create Reason Codes
# ------------------------------------------------------------

content_data['reason_code'] = np.select(
    [
        # High impressions + low CTR
        (content_data['impressions_90d'] >
         content_data['impressions_90d'].median()) &
        (content_data['ctr'] <
         content_data['ctr'].median()),

        # Ranking opportunity
        (content_data['avg_position'] >= 4) &
        (content_data['avg_position'] <= 20),

        # Downward trend
        content_data['trend_direction'].astype(str).str.lower().isin(
            ['down', 'declining', 'decrease']
        ),

        # High search visibility
        content_data['impressions_90d'] >
        content_data['impressions_90d'].median()
    ],
    [
        'HIGH_IMPRESSIONS_LOW_CTR',
        'RANKING_OPPORTUNITY',
        'DOWNWARD_TREND',
        'HIGH_SEARCH_VISIBILITY'
    ],
    default='GENERAL_REVIEW'
)


# ------------------------------------------------------------
# Step 9: Rank all content items
# ------------------------------------------------------------

content_data = content_data.sort_values(
    'action_score',
    ascending=False
).reset_index(drop=True)

content_data['rank'] = (
    content_data.index + 1
)


# ------------------------------------------------------------
# Step 10: Create final ranked queue
# ------------------------------------------------------------

baseline_queue = content_data[
    [
        'content_id',
        'client_id',
        'impressions_90d',
        'clicks_90d',
        'ctr',
        'avg_position',
        'sessions_90d',
        'engagement_rate',
        'trend_direction',
        'trend_pct',
        'action_score',
        'rank',
        'action',
        'reason_code'
    ]
].copy()


# ------------------------------------------------------------
# Step 11: Save the ranked queue
# ------------------------------------------------------------

os.makedirs(
    'work/outputs',
    exist_ok=True
)

output_path = (
    'work/outputs/baseline_action_score.csv'
)

baseline_queue.to_csv(
    output_path,
    index=False
)


# ------------------------------------------------------------
# Step 12: Display results
# ------------------------------------------------------------

print("\n============================================")
print("RANKED QUEUE CREATED SUCCESSFULLY")
print("============================================")

print(
    "Final queue shape:",
    baseline_queue.shape
)

print(
    "\nCSV saved at:"
)

print(output_path)

print(
    "\nAction distribution:"
)

print(
    baseline_queue['action'].value_counts()
)

print(
    "\nTop 20 content items:"
)

display(
    baseline_queue.head(20)
)

Starting dataset shape: (30000, 44)

RANKED QUEUE CREATED SUCCESSFULLY
Final queue shape: (30000, 14)

CSV saved at:
work/outputs/baseline_action_score.csv

Action distribution:
action
Improve    15221
Review     12366
Monitor     2413
Name: count, dtype: int64

Top 20 content items:


,content_id,client_id,impressions_90d,clicks_90d,ctr,avg_position,sessions_90d,engagement_rate,trend_direction,trend_pct,action_score,rank,action,reason_code
0,content_5fe46e04994d,client_4e07408562,517715,741,0.14,4.2,520,4.23,down,-44.8,99.958000,1,Improve,RANKING_OPPORTUNITY
1,content_1a9e894be2e2,client_19581e27de,416180,944,0.23,4.0,1140,2.37,down,-27.0,99.350273,2,Improve,RANKING_OPPORTUNITY
2,content_2c2606c5d176,client_19581e27de,347399,1854,0.53,4.2,2146,1.30,down,-36.5,98.779736,3,Improve,RANKING_OPPORTUNITY
3,content_cb112fce36be,client_19581e27de,309910,492,0.16,5.6,480,2.08,down,-41.8,98.586969,4,Improve,RANKING_OPPORTUNITY
4,content_008fb02c46cb,client_349c41201b,236803,605,0.26,4.4,703,1.14,down,-21.3,97.841251,5,Improve,RANKING_OPPORTUNITY
5,content_c8e9d6ab9013,client_19581e27de,208678,0,0.00,9.7,6,0.00,down,-43.4,97.582914,6,Improve,HIGH_IMPRESSIONS_LOW_CTR
6,content_cea79ef51519,client_f369cb89fc,208798,490,0.23,5.2,244,2.87,down,-35.6,97.515444,7,Improve,RANKING_OPPORTUNITY
7,content_bf7bff5d0756,client_349c41201b,197199,440,0.22,6.8,535,0.75,down,-23.1,97.366407,8,Improve,RANKING_OPPORTUNITY
8,content_9463d30d5826,client_19581e27de,192478,551,0.29,5.7,707,2.83,down,-25.0,97.280948,9,Improve,RANKING_OPPORTUNITY
9,content_3d94572c3a35,client_19581e27de,190623,462,0.24,4.3,441,5.22,down,-52.4,97.270186,10,Improve,RANKING_OPPORTUNITY


## 3. Top-20 review

For each of the top 20: action, reason code, confidence note, and what would make it wrong.

## 3. Top-20 Review

The top 20 content items were reviewed using the baseline action score and its underlying signals. All 20 items received an `Improve` action, indicating that they were prioritized by the baseline scoring rule for further review.

Most of the top-ranked items had high search impressions and average positions between 4 and 10, which created a strong ranking-opportunity signal. Several items also had low CTR or a downward trend, providing additional directional evidence for review.

Confidence in these recommendations is moderate rather than high. The score is based on observed metrics from the available dataset and is intended for decision-support. The recommendations could be wrong if factors not included in the dataset, such as search intent, SERP features, brand effects, content quality, keyword-level performance, or incomplete engagement tracking, explain the observed results.

The table below provides the action, reason code, confidence note, and potential failure condition for each of the top 20 items.


In [33]:
# ============================================================
# 3. TOP-20 REVIEW
# ============================================================

top_20_review = baseline_queue.head(20).copy()

# Add confidence note
top_20_review['confidence_note'] = (
    'Medium confidence: based on observed search visibility, CTR, '
    'ranking position, and available trend signals.'
)

# Add what could make the recommendation wrong
top_20_review['what_would_make_it_wrong'] = (
    'The recommendation may be wrong if search intent, SERP features, '
    'brand effects, keyword mix, content quality, seasonality, or '
    'incomplete tracking explains the observed metrics.'
)

# Select columns required by the assignment
top_20_review = top_20_review[
    [
        'rank',
        'content_id',
        'action',
        'reason_code',
        'confidence_note',
        'what_would_make_it_wrong'
    ]
]

# Display the Top 20 Review
display(top_20_review)

,rank,content_id,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,content_5fe46e04994d,Improve,RANKING_OPPORTUNITY,Medium confidence: based on observed search vi...,The recommendation may be wrong if search inte...
1,2,content_1a9e894be2e2,Improve,RANKING_OPPORTUNITY,Medium confidence: based on observed search vi...,The recommendation may be wrong if search inte...
2,3,content_2c2606c5d176,Improve,RANKING_OPPORTUNITY,Medium confidence: based on observed search vi...,The recommendation may be wrong if search inte...
3,4,content_cb112fce36be,Improve,RANKING_OPPORTUNITY,Medium confidence: based on observed search vi...,The recommendation may be wrong if search inte...
4,5,content_008fb02c46cb,Improve,RANKING_OPPORTUNITY,Medium confidence: based on observed search vi...,The recommendation may be wrong if search inte...
5,6,content_c8e9d6ab9013,Improve,HIGH_IMPRESSIONS_LOW_CTR,Medium confidence: based on observed search vi...,The recommendation may be wrong if search inte...
6,7,content_cea79ef51519,Improve,RANKING_OPPORTUNITY,Medium confidence: based on observed search vi...,The recommendation may be wrong if search inte...
7,8,content_bf7bff5d0756,Improve,RANKING_OPPORTUNITY,Medium confidence: based on observed search vi...,The recommendation may be wrong if search inte...
8,9,content_9463d30d5826,Improve,RANKING_OPPORTUNITY,Medium confidence: based on observed search vi...,The recommendation may be wrong if search inte...
9,10,content_3d94572c3a35,Improve,RANKING_OPPORTUNITY,Medium confidence: based on observed search vi...,The recommendation may be wrong if search inte...


In [34]:
# Show the detailed metrics behind the Top-20 recommendations

top_20_metrics = baseline_queue.head(20)[
    [
        'rank',
        'content_id',
        'impressions_90d',
        'clicks_90d',
        'ctr',
        'avg_position',
        'sessions_90d',
        'engagement_rate',
        'trend_direction',
        'trend_pct',
        'action_score',
        'action',
        'reason_code'
    ]
]

display(top_20_metrics)

,rank,content_id,impressions_90d,clicks_90d,ctr,avg_position,sessions_90d,engagement_rate,trend_direction,trend_pct,action_score,action,reason_code
0,1,content_5fe46e04994d,517715,741,0.14,4.2,520,4.23,down,-44.8,99.958000,Improve,RANKING_OPPORTUNITY
1,2,content_1a9e894be2e2,416180,944,0.23,4.0,1140,2.37,down,-27.0,99.350273,Improve,RANKING_OPPORTUNITY
2,3,content_2c2606c5d176,347399,1854,0.53,4.2,2146,1.30,down,-36.5,98.779736,Improve,RANKING_OPPORTUNITY
3,4,content_cb112fce36be,309910,492,0.16,5.6,480,2.08,down,-41.8,98.586969,Improve,RANKING_OPPORTUNITY
4,5,content_008fb02c46cb,236803,605,0.26,4.4,703,1.14,down,-21.3,97.841251,Improve,RANKING_OPPORTUNITY
5,6,content_c8e9d6ab9013,208678,0,0.00,9.7,6,0.00,down,-43.4,97.582914,Improve,HIGH_IMPRESSIONS_LOW_CTR
6,7,content_cea79ef51519,208798,490,0.23,5.2,244,2.87,down,-35.6,97.515444,Improve,RANKING_OPPORTUNITY
7,8,content_bf7bff5d0756,197199,440,0.22,6.8,535,0.75,down,-23.1,97.366407,Improve,RANKING_OPPORTUNITY
8,9,content_9463d30d5826,192478,551,0.29,5.7,707,2.83,down,-25.0,97.280948,Improve,RANKING_OPPORTUNITY
9,10,content_3d94572c3a35,190623,462,0.24,4.3,441,5.22,down,-52.4,97.270186,Improve,RANKING_OPPORTUNITY


## 4. Weak picks + leakage check

Which picks look wrong and why? Confirm no product flags or future windows leaked in.

In [35]:
# ============================================================
# 4. WEAK PICKS + LEAKAGE CHECK
# ============================================================

# ------------------------------------------------------------
# A. Identify specific potentially weak picks
# ------------------------------------------------------------

top20 = baseline_queue.head(20).copy()

# Weak picks:
# 1. Very low clicks (less than 20)
# 2. High CTR compared with the overall dataset median
weak_picks = top20[
    (top20['clicks_90d'] < 20) |
    (top20['ctr'] > baseline_queue['ctr'].median())
].copy()

print("Potentially weak picks:")
display(
    weak_picks[
        [
            'rank',
            'content_id',
            'impressions_90d',
            'clicks_90d',
            'ctr',
            'avg_position',
            'trend_direction',
            'trend_pct',
            'action_score',
            'action',
            'reason_code'
        ]
    ]
)


# ------------------------------------------------------------
# B. Check for product-related columns
# ------------------------------------------------------------

product_columns = [
    col for col in df.columns
    if 'product' in col.lower()
]

print("\nProduct-related columns found:")
print(product_columns)

if len(product_columns) == 0:
    print("No product-related columns were found.")


# ------------------------------------------------------------
# C. Check for future-date columns
# ------------------------------------------------------------

future_columns = [
    col for col in df.columns
    if any(
        word in col.lower()
        for word in ['future', 'forecast', 'next', 'outcome']
    )
]

print("\nPotential future-related columns found:")
print(future_columns)

if len(future_columns) == 0:
    print("No explicit future-related columns were found.")


# ------------------------------------------------------------
# D. Show the time-window fields
# ------------------------------------------------------------

window_columns = [
    col for col in df.columns
    if any(
        word in col.lower()
        for word in ['90d', '30d', 'prev', 'last', 'trend']
    )
]

print("\nTime-window / trend columns:")
print(window_columns)


# ------------------------------------------------------------
# E. Leakage summary
# ------------------------------------------------------------

print("\nLEAKAGE CHECK SUMMARY")
print("---------------------")
print("Product flags found:", len(product_columns) > 0)
print("Explicit future-related columns found:", len(future_columns) > 0)

print(
    "\nNote: The score uses observed historical metrics and the "
    "precomputed trend_direction field. The construction of "
    "trend_direction and trend_pct should be verified to ensure "
    "they were calculated only from data available at the decision point."
)

Potentially weak picks:


,rank,content_id,impressions_90d,clicks_90d,ctr,avg_position,trend_direction,trend_pct,action_score,action,reason_code
0,1,content_5fe46e04994d,517715,741,0.14,4.2,down,-44.8,99.958000,Improve,RANKING_OPPORTUNITY
1,2,content_1a9e894be2e2,416180,944,0.23,4.0,down,-27.0,99.350273,Improve,RANKING_OPPORTUNITY
2,3,content_2c2606c5d176,347399,1854,0.53,4.2,down,-36.5,98.779736,Improve,RANKING_OPPORTUNITY
3,4,content_cb112fce36be,309910,492,0.16,5.6,down,-41.8,98.586969,Improve,RANKING_OPPORTUNITY
4,5,content_008fb02c46cb,236803,605,0.26,4.4,down,-21.3,97.841251,Improve,RANKING_OPPORTUNITY
5,6,content_c8e9d6ab9013,208678,0,0.00,9.7,down,-43.4,97.582914,Improve,HIGH_IMPRESSIONS_LOW_CTR
6,7,content_cea79ef51519,208798,490,0.23,5.2,down,-35.6,97.515444,Improve,RANKING_OPPORTUNITY
7,8,content_bf7bff5d0756,197199,440,0.22,6.8,down,-23.1,97.366407,Improve,RANKING_OPPORTUNITY
8,9,content_9463d30d5826,192478,551,0.29,5.7,down,-25.0,97.280948,Improve,RANKING_OPPORTUNITY
9,10,content_3d94572c3a35,190623,462,0.24,4.3,down,-52.4,97.270186,Improve,RANKING_OPPORTUNITY



Product-related columns found:
[]
No product-related columns were found.

Potential future-related columns found:
[]
No explicit future-related columns were found.

Time-window / trend columns:
['impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'days_since_last_update', 'trend_direction', 'trend_pct']

LEAKAGE CHECK SUMMARY
---------------------
Product flags found: False
Explicit future-related columns found: False

Note: The score uses observed historical metrics and the precomputed trend_direction field. The construction of trend_direction and trend_pct should be verified to ensure they were calculated only from data available at the decision point.


### Weak Picks + Leakage Check

The baseline ranking is useful for prioritization, but a few Top-20 picks should be treated with extra caution. Rank 6 is a potentially weak pick because it has 208,678 impressions but zero clicks and only 6 sessions. This could indicate a genuine opportunity, but it could also reflect tracking or data-quality issues. Rank 19 is another weak pick because it has 140,079 impressions but only 16 clicks and a 0.01% CTR, so the result should be manually validated before action. Rank 20 may also be a weaker pick for a CTR-focused improvement strategy because it has a relatively high 0.81% CTR. Its high score is mainly supported by its strong downward trend and ranking opportunity.

The leakage check found no product-related columns and no explicit future-related columns in the dataset. The score uses observed performance metrics and the precomputed `trend_direction` field. However, the exact construction of `trend_direction` and `trend_pct` should be verified to confirm that they were calculated only from information available at the decision point. Therefore, the results are treated as directional decision-support rather than guaranteed recommendations.
